# Day 3 — Neuron, Layer, MLP

With `Value` and `backward()` done, build the three layers Karpathy ships in micrograd:

- `Neuron(nin)` — `nin` weights + 1 bias, all `Value` scalars.
- `Layer(nin, nout)` — list of `nout` neurons.
- `MLP(nin, nouts)` — list of layers (e.g. `MLP(3, [4, 4, 1])`).

Then train it on Karpathy's 4-point dataset.

## What you ship today

1. The three classes above.
2. A train loop that drops loss from ~5 to <0.01 in <200 steps.
3. A plot of the loss curve. (If you don't have matplotlib, that's fine — print every 10 steps.)

## Karpathy's training dataset (copy verbatim — it's the lecture's)

In [ ]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]  # target outputs

## Neuron

```
Neuron(nin):
  w = [Value(uniform(-1,1)) for _ in range(nin)]
  b = Value(0.0)             # or Value(uniform(-1,1))
  __call__(x):
    return sum((wi * xi for wi, xi in zip(w, x)), self.b).tanh()
  parameters():
    return self.w + [self.b]
```

Why `Value(uniform(-1, 1))` and not `Value(0)`? Symmetric init breaks ties between neurons in a layer — otherwise every neuron computes the same output and gradient. Standard.

In [ ]:
import random
from value import Value  # or wherever your Value lives

class Neuron:
    pass  # implement

## Train loop

```
for step in range(200):
    ypred = [mlp(x) for x in xs]
    loss = sum((ypred_i - ygt)**2 for ypred_i, ygt in zip(ypred, ys))
    for p in mlp.parameters():
        p.grad = 0
    loss.backward()
    for p in mlp.parameters():
        p.data -= 0.05 * p.grad
    if step % 10 == 0:
        print(step, loss.data)
```

Expected: loss starts around 4-7 (random init), drops below 0.01 by step ~150-200.

**Most common bug:** forgetting `p.grad = 0` at the top of the loop. Without it, gradients from step 1 carry into step 2 etc., and training silently diverges or stalls.

## End-of-day check

- [ ] Loss < 0.01 in < 200 steps with `lr=0.05`.
- [ ] You can predict `ypred` on a fresh input (e.g. `[1.0, 0.0, -1.0]`) and the value makes intuitive sense (continuous, not crazy).
- [ ] You can articulate: what would happen if you set `lr=1.0`? `lr=0.001`? (Big LR: overshoot, loss bounces or NaN. Tiny LR: loss creeps, may need 10000 steps.)

Day 4: pull all of this together into `micrograd_solo.py` + `tests/test_micrograd.py`, with no copy-paste from the notebooks. That's the muscle-memory check.